In [5]:
import random
import pandas as pd
from faker import Faker


fake = Faker('en_NG')

In [18]:
from ipynb.fs.full.procurement import add_inconsistencies

from ipynb.fs.full.procurement import add_nigerian_phone_inconsistencies

In [3]:
# 12 cities repeated to make 25 plants
CITIES = ['Ibadan', 'Lagos', 'Warri', 'Benin', 'Enugu', 
          'Abia', 'Sokoto', 'Kano', 'Kaduna', 'Anambra',
          'Bayelsa', 'Jos']

# Fixed coordinates for each city
CITY_COORDINATES = {
    'Ibadan': (7.3775, 3.9470),
    'Lagos': (6.5244, 3.3792),
    'Warri': (5.5160, 5.7500),
    'Benin': (6.3350, 5.6037),
    'Enugu': (6.5244, 7.5106),
    'Abia': (5.5329, 7.4806),  # Umuahia
    'Sokoto': (13.0059, 5.2476),
    'Kano': (12.0022, 8.5920),
    'Kaduna': (10.5105, 7.4165),
    'Anambra': (6.2209, 6.9366),  # Awka
    'Bayelsa': (4.7719, 6.0699),  # Yenagoa
    'Jos': (9.8965, 8.8583)
}

# Plant types
PLANT_TYPES = [
    'Manufacturing Plant',
    'Spare Parts Warehouse',
    'Distribution Center',
    'Assembly Plant',
    'Regional Service Center'
]

In [12]:
# Generate plant name variations
def generate_plant_name(city, plant_number):
    """Generate descriptive plant names with some Faker variation"""
    
    descriptors = [
        f"{city} Central Warehouse",
        f"{city} Spare Parts Facility",
        f"{city} Distribution Hub",
        f"{city} {fake.company_suffix()} Warehouse",
        f"{city} Industrial Complex",
        f"{city} Manufacturing Center",
        f"{city} Logistics Center",
        f"{city} Regional Depot",
    ]

     # Add plant number if multiple in same city
    name = random.choice(descriptors)
    
    # Add inconsistencies for realism
    if random.random() < 0.3:
        return add_inconsistencies(name, prob=0.5)
    
    return name

In [68]:
def generate_plant_email(city):
    """Generate email based on plant manager's name"""
    if random.random() < 0.1:
        return None

    
    domain = 'brickhouse.com.ng'
    
    email = f"{city}@{domain}"
    
    # Add inconsistencies
    if random.random() < 0.2:
        issues = [
            lambda x: x.upper(),
            lambda x: f" {x}",
            lambda x: f"{x} ",
        ]
        return random.choice(issues)(email)
    
    return email

In [50]:
# Track city counts for numbering
city_counts = {city: 0 for city in CITIES}

# Generate 25 plants
plants = []

# Distribute 25 plants across 12 cities (some cities get multiple)
cities_to_use = CITIES * 3  # Repeat list to have enough
random.shuffle(cities_to_use)
cities_to_use = cities_to_use[:25]  # Take first 25

In [69]:
plants.clear()

In [70]:
for i, city in enumerate(cities_to_use):
    city_counts[city] += 1
    plant_number = city_counts[city]
    
    # Generate plant code
    city_abbr = city[:3].upper()
    plant_code = f"{city_abbr}-{plant_number:02d}"
    
    # Get coordinates
    lat, lng = CITY_COORDINATES[city]
    
    # Add slight variation to coordinates if multiple plants in same city
    if plant_number > 1:
        lat += random.uniform(-0.05, 0.05)
        lng += random.uniform(-0.05, 0.05)

    # GENERATE MANAGER NAME FIRST - before using it in the dictionary
    manager_name = add_inconsistencies(fake.name(), prob=0.25)
    
    plant = {
        'plant_id': f'LCN{i+1:03d}',
        'plant_code': plant_code if random.random() > 0.1 else add_inconsistencies(plant_code, prob=0.5),
        'plant_name': generate_plant_name(city, plant_number),
        'address': add_inconsistencies(fake.address(), prob=0.3),
        'state': city if random.random() > 0.15 else add_inconsistencies(city, prob=0.6),
        'country': 'Nigeria',
        'latitude': round(lat, 6),
        'longitude': round(lng, 6),
        'plant_type': random.choice(PLANT_TYPES),
        'plant_manager_name': manager_name,
        'phone_number': add_nigerian_phone_inconsistencies(),
        'contact_email': generate_plant_email(city)
    }
    
    plants.append(plant)

In [71]:
# Create datasets and save to CSV
df_plants = pd.DataFrame(plants)
df_plants.to_csv('location.csv', index=False)